# ARCHS4 CLAMP models with BP prior at 1% sample coverage

**Environment:** `clamp-analyses`  

This notebook builds 3 CLAMP models using the BP (GO Biological Process) prior at 1% sample coverage, each with a different random seed for sample shuffling.

Steps (repeated for each seed):
1. Subsample the preprocessed ARCHS4 data with study-based random selection
2. Compute SVD on the subsampled data
3. Run CLAMPbase
4. Run CLAMPfull with BP prior

Each run uses a different seed (`base_seed + 0:2`) for reproducibility, allowing comparison of model stability across different sample selections.

## Load libraries

In [1]:
if (!requireNamespace("CLAMP", quietly = TRUE)) {
    REPO_PATH <- "/home/msubirana/Documents/pivlab/CLAMP" 
    remotes::install_local(REPO_PATH, force = TRUE, dependencies = FALSE)
}

library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(CLAMP)
library(rhdf5)

source(here("config.R"))


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: Matrix

Loaded glmnet 4.1-10

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses



## Configuration

In [ ]:
# Base output directory
base_output_dir <- config$ARCHS4$DATASET_FOLDER

# Coverage level for this notebook
coverage <- 0.01
coverage_pct <- coverage * 100

# Data path for priors
data_path <- here::here('data/archs4')
archs4_file <- here('data/archs4/human_gene_v2.5.h5')

# SVD parameters
N_CORES <- config$ARCHS4$CLAMP_PARAMS$RANDOM_SVD_N_CORES

base_seed <- config$ARCHS4$CLAMP_PARAMS$RANDOM_SVD_SEED
seeds <- base_seed + 0:2
n_runs <- length(seeds)
message("Will run ", n_runs, " models with seeds: ", paste(seeds, collapse = ", "))

Will run 3 models with seeds: 123, 124, 125



## Load preprocessed data

In [3]:
# Load metadata
meta <- readRDS(file.path(base_output_dir, "metadata_filtered.rds"))
n_genes_thin <- meta$n_genes_thin
n_samples_total <- meta$n_samples
archs4_genes <- meta$gene_symbols_thin

# Load sample names
all_samples <- readRDS(file.path(base_output_dir, "all_samples.rds"))
sample_names_total <- all_samples[seq_len(n_samples_total)]

# Load BP pathway
BP_pathMat <- readRDS(file.path(data_path, "BP_pathMat.rds"))
BP_matched <- getMatchedPathwayMat(BP_pathMat, archs4_genes)
message("Loaded and matched BP pathway matrix")

# Save matched BP matrix
bp_matched_out <- file.path(base_output_dir, "BP_matched.rds")
saveRDS(BP_matched, bp_matched_out, compress = "xz")
message("Saved BP_matched to: ", bp_matched_out)

# Load FBM
fbm_file <- file.path(base_output_dir, "fbm")
output_file <- paste0(fbm_file, "_filtered")

archs4_fbm_filt <- FBM(
  nrow        = n_genes_thin,
  ncol        = n_samples_total,
  backingfile = output_file,
  create_bk   = FALSE
)

message("Loaded FBM with ", n_genes_thin, " genes and ", n_samples_total, " samples")

There are 11936 genes in the intersection between data and prior

Removing 2059 pathways

Loaded and matched BP pathway matrix

Saved BP_matched to: /home/msubirana/Documents/pivlab/clamp-analyses/output/archs4/BP_matched.rds

Loaded FBM with 18423 genes and 605614 samples



## Run 3 CLAMP models with different seeds

In [8]:
# Get series information once (shared across all runs)
message("Getting series information from ARCHS4 HDF5 file...")

geo_acc <- h5read(archs4_file, "/meta/samples/geo_accession")
series_id <- h5read(archs4_file, "/meta/samples/series_id")

gsm_to_gse <- data.frame(
  geo_accession = geo_acc,
  series_id = series_id,
  stringsAsFactors = FALSE
)

# Create mapping of our filtered samples to series
sample_series_df <- data.frame(
  sample_idx = seq_len(n_samples_total),
  sample_name = sample_names_total,
  stringsAsFactors = FALSE
) %>%
  left_join(gsm_to_gse, by = c("sample_name" = "geo_accession"))

# Get unique series and their sample counts
series_counts <- sample_series_df %>%
  group_by(series_id) %>%
  summarise(n_samples = n(), .groups = "drop") %>%
  arrange(series_id)

message("Found ", nrow(series_counts), " unique studies")

n_genes <- nrow(archs4_fbm_filt)
n_samples_target <- round(n_samples_total * coverage)
message("Target samples per run: ", n_samples_target, " (", coverage_pct, "% of ", n_samples_total, ")")

# Store results summary
results_summary <- data.frame(
  run = integer(),
  seed = integer(),
  n_samples = integer(),
  n_studies = integer(),
  CLAMP_K = integer(),
  stringsAsFactors = FALSE
)

for (run_idx in seq_len(n_runs)) {
  current_seed <- seeds[run_idx]
  message("\n", strrep("=", 60))
  message("RUN ", run_idx, "/", n_runs, " - Seed: ", current_seed)
  message(strrep("=", 60))
  
  # Create output directory for this run
  output_dir <- file.path(base_output_dir, paste0("bp_coverage_", coverage_pct, "_seed_", run_idx))
  dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)
  
  # Sample selection with current seed
  set.seed(current_seed)
  shuffled_series <- sample(series_counts$series_id)
  
  selected_samples <- c()
  selected_series <- c()
  n_collected <- 0
  
  for (s in shuffled_series) {
    if (n_collected >= n_samples_target) break
    series_samples <- sample_series_df$sample_idx[sample_series_df$series_id == s]
    n_series_samples <- length(series_samples)
    
    # Skip if no samples found
    if (n_series_samples == 0) {
      warning("No samples found for series: ", s)
      next
    }

    if (n_collected + n_series_samples <= n_samples_target) {
      selected_samples <- c(selected_samples, series_samples)
      selected_series <- c(selected_series, s)
      n_collected <- n_collected + n_series_samples
    } else {
      n_needed <- n_samples_target - n_collected
      selected_samples <- c(selected_samples, series_samples[1:n_needed])
      selected_series <- c(selected_series, s)
      n_collected <- n_collected + n_needed
      break
    }
  }
  
  sample_idx <- sort(selected_samples)
  n_samples <- length(sample_idx)
  sample_names <- sample_names_total[sample_idx]
  
  message("Selected ", length(selected_series), " studies with ", n_samples, " total samples")
  
  # Save sample info
  saveRDS(list(
    run = run_idx,
    seed = current_seed,
    coverage = coverage,
    n_samples = n_samples,
    sample_idx = sample_idx,
    sample_names = sample_names,
    selected_series = selected_series,
    sampling_method = "study_based"
  ), file = file.path(output_dir, "subsample_info.rds"))
  
  # Create subsampled FBM using big_copy for better performance
  message("Creating subsampled FBM via big_copy...")
  fbm_sub_file <- file.path(output_dir, "fbm_subsampled")

  Y_sub <- big_copy(
    archs4_fbm_filt,
    ind.col = sample_idx,
    backingfile = fbm_sub_file
  )

  # SVD
  message("Computing SVD...")
  SVD_K <- min(n_samples - 1, n_genes - 1)
  
  if (N_CORES > 1) {
    options(bigstatsr.check.parallel.blas = FALSE)
    blas_nproc <- getOption("default.nproc.blas")
    options(default.nproc.blas = NULL)
  }
  
  svd_result <- big_randomSVD(Y_sub, k = SVD_K, ncores = N_CORES)
  
  if (N_CORES > 1) {
    options(bigstatsr.check.parallel.blas = TRUE)
    options(default.nproc.blas = blas_nproc)
  }
  
  valid_idx <- which(!is.nan(svd_result$d))
  svd_result$d <- svd_result$d[valid_idx]
  svd_result$u <- svd_result$u[, valid_idx, drop = FALSE]
  svd_result$v <- svd_result$v[, valid_idx, drop = FALSE]
  
  saveRDS(svd_result, file = file.path(output_dir, "svd.rds"))
  
  # Estimate CLAMP K
  svd_list <- list(d = svd_result$d)
  CLAMP_K <- num.pc(svd_list) * 2
  message("CLAMP K = ", CLAMP_K)
  saveRDS(CLAMP_K, file = file.path(output_dir, "CLAMP_K.rds"))
  
  # CLAMPbase
  message("Running CLAMPbase...")
  baseRes <- CLAMPbase(
    Y = Y_sub,
    svdres = svd_result,
    trace = TRUE,
    clamp_k = CLAMP_K
  )
  
  baseRes$Z <- data.frame(baseRes$Z)
  rownames(baseRes$Z) <- archs4_genes
  baseRes$B <- data.frame(baseRes$B)
  colnames(baseRes$B) <- sample_names
  
  saveRDS(baseRes, file = file.path(output_dir, "CLAMPbase.rds"))
  
  model_dir <- file.path(output_dir, "CLAMPbase")
  dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
  write.csv(baseRes$B, file.path(model_dir, "B.csv"))
  write.csv(baseRes$Z, file.path(model_dir, "Z.csv"))
  
  # CLAMPfull with BP prior
  message("Running CLAMPfull with BP prior...")
  fullRes <- CLAMPfull(
    Y = Y_sub,
    svdres = svd_result,
    priorMat = BP_matched,
    clamp.base.result = baseRes,
    use_cpp = TRUE,
    trace = TRUE,
    clamp_k = CLAMP_K
  )
  
  fullRes$Z <- data.frame(fullRes$Z)
  rownames(fullRes$Z) <- archs4_genes
  fullRes$B <- data.frame(fullRes$B)
  colnames(fullRes$B) <- sample_names
  fullRes$summary <- fullRes$summary %>%
    dplyr::rename(LV = LV_index) %>%
    dplyr::mutate(LV = paste0('LV', LV))
  
  saveRDS(fullRes, file = file.path(output_dir, "CLAMPfull_BP.rds"))
  
  model_dir <- file.path(output_dir, "CLAMPfull_BP")
  dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
  write.csv(fullRes$B, file.path(model_dir, "B.csv"))
  write.csv(fullRes$Z, file.path(model_dir, "Z.csv"))
  write.csv(fullRes$summary, file.path(model_dir, "summary.csv"))
  
  # Store summary
  results_summary <- rbind(results_summary, data.frame(
    run = run_idx,
    seed = current_seed,
    n_samples = n_samples,
    n_studies = length(selected_series),
    CLAMP_K = CLAMP_K
  ))
  
  # Clean up memory
  rm(Y_sub, svd_result, baseRes, fullRes)
  gc()
}

message("\n", strrep("=", 60))
message("All ", n_runs, " runs completed!")
message(strrep("=", 60))

Getting series information from ARCHS4 HDF5 file...



Found 18084 unique studies

Target samples per run: 6056 (1% of 605614)



RUN 1/3 - Seed: 123


Selected 220 studies with 6056 total samples

Creating subsampled FBM via big_copy...

Computing SVD...

